# 00 — Comprendre Sharm Image

**Objectif :** comprendre les composants et retrouver chaque étape sans confondre Base et Monster.

Lire les notebooks dans l'ordre indiqué dans le [sommaire](README.md). Les cellules de lecture utilisent le CPU. Les téléchargements, calculs GPU, entraînements et envois sont désactivés par défaut. Le code d'apprentissage reste dans les scripts versionnés, appelés depuis les notebooks.

Ce parcours est préparé pendant le calcul distant. Les résultats finaux apparaîtront après leur collecte et leur archivage ; une donnée absente reste absente.

In [ ]:
from pathlib import Path
import sys

book_dir = next(
    folder
    for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent / "notebooks", parent / "2-LoRA-qwen/notebooks", parent)
    if (folder / "notebook_utils.py").is_file()
)
if str(book_dir) not in sys.path:
    sys.path.insert(0, str(book_dir))
from notebook_utils import (
    PROJECT, REPO, ARTIFACTS, read_json, read_jsonl, sha256,
    contained, current_dataset, archived_report, result_file,
    show_json, table, show_image, run_script,
)
print("Projet :", PROJECT.name)


## Deux modèles, deux rôles

| Nom | Contenu | Ce qui apprend |
|---|---|---|
| **Sharm Image Base** | Qwen Image 2.1, VL 4B FP8, projecteur, diffuseur et VAE | Projecteur spatial ; LoRA du diffuseur facultatif |
| **Sharm Image Monster Pixel Art** | Base précise + adaptateur Monster | LoRA du diffuseur, lié à l'empreinte du projecteur |

Le **8B BF16** sert d'enseignant hors ligne. Il n'est pas inclus dans la Base 4B.

~~~text
Texte ──► VL 4B FP8 gelé ──► projecteur entraînable ──► diffuseur ──► VAE ──► PNG RGBA
                                  ▲                       ▲
                    8B BF16 / cibles spatiales       LoRA Monster ensuite
~~~

Les paramètres FP8 gelés ne reçoivent pas de gradients. Les parties entraînables utilisent le calcul BF16 et des poids maîtres FP32. Un fichier PNG possède un format ; la transparence demande aussi un canal alpha correctement généré.

In [ ]:
dataset, pointer = current_dataset()
info = read_json(dataset / "dataset.json", {})
table([
    {"élément": "Version du dataset Monster", "valeur": dataset.name},
    {"élément": "Images uniques", "valeur": info.get("unique_images", "absent")},
    {"élément": "Espèces", "valeur": info.get("species", "absent")},
    {"élément": "Audit passé", "valeur": read_json(dataset / "audit.json", {}).get("passed", False)},
    {"élément": "Entraînement déclaré dans cette préparation", "valeur": pointer.get("training_started", False)},
], ["élément", "valeur"])

## Lire le résultat final

Le rapport sélectionné par `reports/current.json` est une archive versionnée d'un run terminé, revu et publié. Il prime sur les fichiers locaux temporaires. Avant cet archivage, les notebooks peuvent lire les petits rapports locaux disponibles, datés ; ils ne contactent pas implicitement le serveur.

Les chapitres suivants séparent la perte numérique, la revue visuelle et l'évaluation humaine statistique. Aucune de ces mesures ne doit être présentée comme une garantie de positions parfaites.

In [ ]:
archive = archived_report()
if archive:
    show_json(archive / "summary.json")
else:
    print("Aucun rapport spatial final versionné pour le moment.")
    observation = read_json(ARTIFACTS / "spatial/remote-observation.json", {})
    table([{
        "relevé UTC": observation.get("checked_at", "indisponible"),
        "phase": observation.get("run", {}).get("stage", "indisponible"),
        "pas": observation.get("flow", {}).get("step", "indisponible"),
    }], ["relevé UTC", "phase", "pas"])

## Version du code et du kernel

Les versions ci-dessous décrivent le kernel qui lit ce notebook. Les identités et SHA-256 du run archivé décrivent le calcul réellement effectué. Ce sont deux informations distinctes.

In [ ]:
import importlib.metadata
import platform
import subprocess

print("Python :", platform.python_version())
print("Git :", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
versions = []
for name in ("torch", "transformers", "diffusers", "bitsandbytes", "pillow", "numpy", "IPython"):
    try:
        version = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        version = "non installé dans ce kernel"
    versions.append({"bibliothèque": name, "version": version})
table(versions, ["bibliothèque", "version"])

**Suite :** [01 — Dataset Monster](01_dataset_monster.ipynb).

Les poids et environnements restent hors Git. Les notebooks, scripts, paramètres, PNG sources, données validées, définitions et rapports finaux sont versionnés. Les révisions HF immuables identifient les poids publiés.